# AVA-VLA FFT · canonical D · Suite-balanced · CU 예산

새 Run ID에서 실행합니다. 공유 소스·TAR·T0와 기존 노트북을 수정하지 않습니다.
Train=Suite-balanced(4 suite 각 25%, 부족한 suite replacement), Validation=Episode입니다.
Test는 무결성 검사 외에는 사용하지 않습니다. AVA 학습은 기존 **zero-context FFT**이며
temporal 학습이나 원 논문 재현을 의미하지 않습니다. Cache/T1/Student/양자화는 포함하지 않습니다.

기본값은 준비 및 학습을 실행하지 않습니다. ① 설정/Drive 점검 → ② PREPARE_INPUTS=True로
로컬 검증·Python 3.11 구성 → ③ CU를 기록하고 RUN_PILOT=True로 4-step 실행 →
④ 파일럿 후 CU와 현재 CU를 입력 → ⑤ 예산을 읽고 RUN_MAIN_FFT=True로 실행합니다.
파일럿과 본 학습은 별도 프로세스이며 본 학습은 항상 T0 또는 명시한 본 학습 checkpoint에서 시작합니다.

CU 공식 잔액 API를 사용하지 않습니다. 사용자가 Colab UI의 잔액을 입력합니다.
5 CU 잔여는 실측 기반 추정이며 보장되지 않습니다. CU 소모율이 없으면 명시적 시간 상한이 필요하고,
그 모드에서는 5 CU 목표를 제어할 수 없음을 보고합니다.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, math, os, shutil, subprocess, sys, time, uuid
SESSION_STARTED_UNIX = globals().get('SESSION_STARTED_UNIX', time.time())
PROJECT_ROOT = Path('/content/vlade_work')
OUTPUT_ROOT = Path('/content/vlade_runs/ava_fft_suite_balanced_seed42')
RUN_ID = globals().get('RUN_ID', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])
# 재시작 후에는 기존 Run ID와 RESUME_CHECKPOINT를 명시합니다. 자동 latest 선택 없음.
RESUME_CHECKPOINT = None
PREPARE_INPUTS = False
PREPARATION_CACHE = OUTPUT_ROOT / 'verified_inputs'
LOCAL_VALIDATION_RECEIPTS = Path('/content/vlade_verified_input_receipts')
# Inputs are pinned in configs/storage/hf_assets.lock.json.

RUN_PILOT = False
RUN_MAIN_FFT = False
SAVE_STEPS = 2000
SAVE_OPTIMIZER_STATE = True
AUTO_UNASSIGN_RUNTIME = True
CURRENT_COMPUTE_UNITS = None  # 본 학습 직전 Colab UI에서 갱신
TARGET_REMAINING_CU = 5.0
CU_BEFORE_PILOT = None
CU_AFTER_PILOT = None
# 공식적으로 확인 가능한 세션 종료 Unix 시각을 아는 경우만 입력. 고정 8시간 가정 없음.
SESSION_DEADLINE_UNIX = None
EXPLICIT_MAIN_TIME_LIMIT_SECONDS = None  # CU 실측 불가 시 필수인 사용자 시간 상한
PILOT_TIME_LIMIT_SECONDS = None  # 4-step도 명시적 상한 필요; 모델 로딩/저장/재로딩 포함
# Checkpoint space is checked on the runtime SSD.
TAR_PATH = 'hfasset://prepared_d_seed42'
EXPECTED_TAR_BYTES = 11_371_123_200
EXPECTED_TAR_SHA256 = None  # 독립적으로 검증된 Manifest에서만 채웁니다. 실측값을 기대값으로 복사 금지
SUITE_MAP_ASSET = 'suite_map_seed42'
MAX_EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 8
RUN_DIR = OUTPUT_ROOT / RUN_ID
LOCAL = Path('/content') / ('vlade-suite-' + RUN_ID)
CODE = LOCAL / 'code'
print(json.dumps({'GPU/VRAM': '다음 셀에서 확인', 'project': str(PROJECT_ROOT),
 'TAR': str(TAR_PATH), 'teacher': 'hfasset://ava_t0',
 'dataset': 'LIBERO+ canonical D 80/10/10 seed42', 'episodes': [11477,1428,1442],
 'train_queries':228908, 'train_sampling':'suite_balanced', 'validation_sampling':'episode',
 'suite_target':[.25,.25,.25,.25], 'current_CU':CURRENT_COMPUTE_UNITS,
 'target_remaining_CU':TARGET_REMAINING_CU, 'measured_CU_rate':None, 'possible_steps':None,
 'estimated_runtime_seconds':None, 'checkpoint_bytes':None, 'save_steps':SAVE_STEPS,
 'safe_reserve_seconds':None, 'RUN_PILOT':RUN_PILOT, 'RUN_MAIN_FFT':RUN_MAIN_FFT,
 'session_deadline_unix':SESSION_DEADLINE_UNIX,'explicit_main_seconds':EXPLICIT_MAIN_TIME_LIMIT_SECONDS,
 'tar_expected_sha256':EXPECTED_TAR_SHA256,'tar_integrity':'미검증',
 'actual_training_started':False, 'output':str(RUN_DIR)}, indent=2, ensure_ascii=False))

## Drive와 실제 경로
TAR ID `17Nim7jPu0dC-wKNkNjAtPyUJAre4QYnM`, 11,371,123,200 bytes를 조회했습니다.
실제 부모는 `data/splits`입니다. 중첩된 공유 압축 해제 폴더는 사용하지 않습니다.
TAR 기대 SHA256은 아직 확보하지 못했습니다. 실측 SHA256 기록만으로 검증 완료라고 하지 않습니다.
대신 로컬 보관 canonical D의 고정 manifest/split/normalization 해시 및 모든 payload를 대조하는
동등 무결성 경로를 사용합니다. 어느 하나라도 불일치하면 학습을 중단합니다.

In [ ]:
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True, capture_output=True)
print('GPU / VRAM:', gpu.stdout.strip() or gpu.stderr.strip())
print('HF input preparation:', PREPARE_INPUTS)

## 검증된 파일 목록만 Colab 로컬로 준비
공유 source manifest SHA를 먼저 검증합니다. 최근 Suite-balanced 변경 두 파일은
2026-11-04에 Drive에서 읽은 SHA256으로 별도 고정합니다. 기존 source manifest는 수정하지 않습니다.
신규 어댑터는 이 노트북에 내장되어 로컬에만 쓰이며 아래 embedded SHA로 검증합니다.
기존 `fft.py`, `prepared_fft.py`, `episode_data.py`를 덮어쓰지 않습니다.

In [ ]:
READY = False
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()
def control_write(path, value):
    path.parent.mkdir(parents=True,exist_ok=True)
    raw = json.dumps(value,indent=2,ensure_ascii=False,allow_nan=False,default=str)
    temp = path.with_suffix(path.suffix+'.tmp')
    with temp.open('w') as f:
        f.write(raw); f.flush(); os.fsync(f.fileno())
    temp.replace(path)
    assert json.loads(path.read_text()) == json.loads(raw)
def run(args, env=None):
    command=[str(x) for x in args]
    started=time.monotonic()
    label=Path(command[0]).name+' '+(Path(command[1]).name if len(command)>1 else '')
    print('[명령 시작]',label,flush=True)
    output=[]
    process=subprocess.Popen(command,cwd=CODE,env=env or ENV,text=True,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True);output.append(line)
    returncode=process.wait();text=''.join(output)
    print('[명령 종료]',label,f'{time.monotonic()-started:.1f}s',flush=True)
    if returncode:
        if (RUN_DIR/'logs').is_dir():
            with (RUN_DIR/'logs/errors.log').open('a') as stream:stream.write(text)
        raise RuntimeError(f'하위 프로세스 실패 (exit={returncode}): {label}')
    return subprocess.CompletedProcess(command,returncode,text,'')

import re, tempfile
def stage_source(repository: str, scratch_root: Path, source_ref: str = "main") -> Path:
    """Resolve a ref once and return a clean checkout of that immutable commit.

    Existing checkouts are never updated: another process may be using them.
    Git checks object integrity, so no per-file SHA256 inventory is needed.
    """
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./-]*", source_ref) or ".." in source_ref:
        raise ValueError("Invalid Git source reference")
    scratch_root = scratch_root.resolve()
    scratch_root.mkdir(parents=True, exist_ok=True)

    def git(directory, *args):
        return subprocess.run(["git", "-C", str(directory), *args], check=True,
                              capture_output=True, text=True).stdout.strip()

    with tempfile.TemporaryDirectory(prefix="vlade-fetch-", dir=scratch_root) as temporary:
        staging = Path(temporary) / "source"
        git(scratch_root, "clone", "--no-checkout", "--filter=blob:none", "--", repository, str(staging))
        git(staging, "fetch", "--depth", "1", "origin", source_ref)
        commit = git(staging, "rev-parse", "FETCH_HEAD^{commit}")
        # Keep submodule metadata paths short on Windows; check the full SHA below.
        destination = scratch_root / ("vlade_colab_" + commit[:16])
        if destination.exists():
            if (git(destination, "rev-parse", "HEAD") != commit
                    or git(destination, "remote", "get-url", "origin") != repository
                    or git(destination, "status", "--porcelain", "--untracked-files=normal")
                    or any(line[:1] in {"-", "+", "U"} for line in git(destination, "submodule", "status").splitlines()
                           if "third_party/ava_vla" in line)):
                raise ValueError("Existing source checkout changed; use a fresh source cache")
            return destination
        git(staging, "checkout", "--detach", commit)
        # The parent commit pins the AVA gitlink. Other baselines use their own environments.
        if (staging / ".gitmodules").is_file():
            git(staging, "submodule", "update", "--init", "--depth", "1", "--", "third_party/ava_vla")
        staging.rename(destination)
    return destination

if PREPARE_INPUTS:
    if RUN_DIR.exists() and RESUME_CHECKPOINT is None:
        raise FileExistsError('Use a new run ID or select an explicit resume checkpoint')
    CODE = stage_source('https://github.com/eastha10/VLADE.git', Path('/content/vlade_sources'),
                        globals().get('SOURCE_REF', 'main'))
    SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
    PROJECT_ROOT = CODE
    for folder in ['configs','pilot4','logs','manifests','reports']:
        (RUN_DIR/folder).mkdir(parents=True,exist_ok=True)
    LOCAL.mkdir(parents=True,exist_ok=True)
    (LOCAL/'tmp').mkdir(exist_ok=True)
    ENV = os.environ.copy()
    ENV.update(PYTHONPATH=str(CODE/'src'), PYTHONDONTWRITEBYTECODE='1',
        TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
        UV_CACHE_DIR=str(LOCAL/'uv_cache'), TMPDIR=str(LOCAL/'tmp'), XDG_CACHE_HOME=str(LOCAL/'cache'))
    control_write(RUN_DIR/'manifests/source_manifest.json', {'git_commit':SOURCE_COMMIT})
    sys.path.insert(0,str(CODE/'src'))
    # Git supplies source. HF supplies every external model/data input.
    ASSET_CACHE = Path('/content/vlade_hf_assets')
    ASSET_WORK_ROOT = Path('/content/vlade_asset_work')
    RUNS_ROOT = Path('/content/vlade_runs')
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    _HF_FETCHED = set()
    ENV.update(VLADE_ASSET_CACHE=str(ASSET_CACHE),
               VLADE_ASSET_REGISTRY=str(CODE/'configs/storage/hf_assets.lock.json'),
               HF_HOME='/content/vlade_hf_home', HF_TOKEN_PATH='/content/vlade_hf_token')
    os.environ.update({key: ENV[key] for key in ('VLADE_ASSET_CACHE','VLADE_ASSET_REGISTRY')})

    def hf_input(key, python=None, *, fetch=True):
        from vlade.storage.assets import asset_record, cache_path
        record = asset_record(key, Path(ENV['VLADE_ASSET_REGISTRY']))
        target = cache_path(key, record, ASSET_CACHE)
        if fetch and (key, str(target)) not in _HF_FETCHED:
            if python is None:
                raise ValueError('Select the CPU/model Python environment before fetching HF assets')
            run([python, 'scripts/data/hf_assets.py', 'fetch', key,
                 '--registry', ENV['VLADE_ASSET_REGISTRY'], '--cache', ASSET_CACHE], env=ENV)
            _HF_FETCHED.add((key, str(target)))
        return target

    def hf_path(value, python=None, *, fetch=True):
        value = str(value)
        if not value.startswith('hfasset:'):
            raise ValueError('External inputs must use hfasset:// keys, not Drive/local paths')
        return hf_input(value[len('hfasset:'):].strip('/\\'), python, fetch=fetch)

In [ ]:
# HF 설정 후 기존 학습 셀을 실행하면 checkpoint 저장 시 자동 전송합니다.
# 저장소는 HF 웹에서 Private로 먼저 만들고 해당 저장소 쓰기 토큰을 Secrets에 등록하십시오.
if 'ENV' not in globals():
    ENV = os.environ.copy()
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
ENV['HF_TOKEN'] = os.environ['HF_TOKEN']
import datetime, uuid
ENABLE_HUB_UPLOAD = globals().get('ENABLE_HUB_UPLOAD', True)
HF_REPO_ID = 'VLADE-konkuk/VLADE-checkpoints'  # 조직 저장소; 이전 런타임 값도 갱신합니다.
HF_RUN_GROUP = globals().get('HF_RUN_GROUP', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])  # 재개 시 기존 group 지정
if ENABLE_HUB_UPLOAD:
    if not HF_RUN_GROUP:
        raise ValueError('HF_RUN_GROUP을 지정하십시오. pilot/main은 출력 폴더명으로 구분합니다.')
    os.environ['VLADE_HUB_REPO'] = HF_REPO_ID
    os.environ['VLADE_HUB_RUN_ID'] = HF_RUN_GROUP
    os.environ['HF_TOKEN_PATH'] = '/content/vlade_hf_token'
    # 모델 캐시는 기존 위치를 유지하고 토큰 파일은 런타임 로컬에만 둡니다.
    ENV.update(HF_TOKEN=os.environ['HF_TOKEN'], VLADE_HUB_REPO=HF_REPO_ID,
               VLADE_HUB_RUN_ID=HF_RUN_GROUP, HF_TOKEN_PATH='/content/vlade_hf_token')
    print('Private Hub 업로드 설정:', HF_REPO_ID, HF_RUN_GROUP)
else:
    for key in ('VLADE_HUB_REPO', 'VLADE_HUB_RUN_ID'):
        ENV.pop(key, None)
        os.environ.pop(key, None)
    print('Hub 업로드 꺼짐: HF_RUN_GROUP 설정 후 ENABLE_HUB_UPLOAD=True로 이 셀을 다시 실행하십시오.')

In [ ]:
if PREPARE_INPUTS:
    assert (CODE/'src/vlade/training/colab_suite_fft.py').is_file()


## AVA Python 3.11 격리 환경
첨부 노트북의 고정 Transformers fork/patch/wheel SHA 검사를 재사용합니다.
패키지는 Colab 로컬에만 설치하고 공유 upstream에는 쓰지 않습니다.
설치 도구를 먼저 확인하며 실패 시 실제 stdout/stderr를 출력합니다.

In [ ]:
AVA_ENV_READY = False
if PREPARE_INPUTS:
    UV = shutil.which('uv')
    if UV is None:
        # 현재 Python은 설치 도구 bootstrap에만 사용; AVA 실행에는 사용하지 않습니다.
        bootstrap = LOCAL/'uv-tool'
        run([sys.executable,'-m','venv',bootstrap])
        tool_python = bootstrap/'bin/python'
        probe = subprocess.run([str(tool_python),'-m','pip','--version'],capture_output=True,text=True)
        if probe.returncode:
            print(probe.stdout,probe.stderr)
            run([tool_python,'-m','ensurepip','--upgrade'])
        run([tool_python,'-m','pip','install','uv'])
        UV = str(bootstrap/'bin/uv')
    run([UV,'--version'])
    def environment(name, version, packages):
        directory = Path('/content')/name
        if not (directory/'bin/python').is_file():run([UV,'venv','--python',version,directory])
        python = directory/'bin/python'
        run([UV,'pip','install','--python',python,*packages])
        return python

In [ ]:
if PREPARE_INPUTS:
    AVA_ENV_READY = False
    AVA_PY = None
    import tempfile, tomllib, zipfile
    AVA_VENV = Path('/content') / ('vlade_ava_fft_py311_' + RUN_ID)
    AVA_PATCH_REL = 'configs/models/teacher/transformers-avavla-a03eee5-checkpoint-extra-attn.patch'
    AVA_PATCH_SHA = '9f0e46c9e93d62a92af950aeee712191cfcf314c1e636a9608fddcb57ff6087d'
    AVA_TRANSFORMERS_REV = 'a03eee5da1794c45f5a911b60a9a4545e6e0ed1f'
    AVA_BASE_SHA = '62f22856fe7b8459d6d61d4676b0075b65724979f84859ac475dbe36c10a3d80'
    AVA_INSTALLED_SHA = 'aeca129d863bec439623f54d93810792d82c86921c25fbe69d9169ffdb373788'
    AVA_VERIFY_SOURCE = "import ast, hashlib, importlib, json, sys\nfrom pathlib import Path\nexpected_sha, expected_prefix = sys.argv[1:3]\nprefix = Path(sys.prefix).resolve()\nif sys.prefix == sys.base_prefix or prefix != Path(expected_prefix).resolve():\n    raise RuntimeError('AVA verification requires the dedicated virtual environment')\nmodule = importlib.import_module('transformers.models.llama.modeling_llama')\nsource = Path(module.__file__).resolve()\nif not source.is_relative_to(prefix) or source.name != 'modeling_llama.py':\n    raise RuntimeError('Transformers was imported outside the AVA environment')\npayload = source.read_bytes()\nif hashlib.sha256(payload).hexdigest() != expected_sha:\n    raise RuntimeError('Installed AVA Transformers source SHA mismatch')\ntree = ast.parse(payload)\nmodel = next(node for node in tree.body if isinstance(node, ast.ClassDef) and node.name == 'LlamaModel')\nforward = next(node for node in model.body if isinstance(node, ast.FunctionDef) and node.name == 'forward')\ncalls = [node for node in ast.walk(forward) if isinstance(node, ast.Call)\n         and isinstance(node.func, ast.Attribute) and node.func.attr == '_gradient_checkpointing_func']\nexpected = 'extra_attn_weights if extra_layer_ids and layer_idx in extra_layer_ids else None'\nif len(calls) != 1 or len(calls[0].args) != 9 or ast.unparse(calls[0].args[-1]) != expected:\n    raise RuntimeError('Installed checkpoint attention argument was not preserved')\nprint(json.dumps({'source': str(source), 'sha256': expected_sha, 'venv': str(prefix)}))"
    _VLADE_AVA_BASE_RUN = getattr(run, '_vlade_base_run', run)

    def verify_ava_transformers(python):
        _VLADE_AVA_BASE_RUN([python, '-c', AVA_VERIFY_SOURCE, AVA_INSTALLED_SHA, AVA_VENV])

    def require_ava_environment():
        if not AVA_ENV_READY or AVA_PY is None:
            raise RuntimeError('AVA patched environment is not verified; rerun the installation cell')
        verify_ava_transformers(AVA_PY)

    def run(args, env=None):
        command = [str(value) for value in args]
        if command and (command[0] == str(AVA_VENV / 'bin/python')
                        or any(Path(value).name == 'run_fft.py' for value in command)):
            require_ava_environment()
        return _VLADE_AVA_BASE_RUN(args, env=env)

    run._vlade_base_run = _VLADE_AVA_BASE_RUN
    patch = CODE / AVA_PATCH_REL
    if not patch.is_file() or hashlib.sha256(patch.read_bytes()).hexdigest() != AVA_PATCH_SHA:
        raise RuntimeError('Staged AVA patch is missing or has the wrong SHA')

    build_root = Path(tempfile.mkdtemp(prefix='vlade-ava-transformers-', dir=LOCAL))
    source = build_root / 'source'
    source.mkdir()
    build_env = ENV.copy()
    build_env.update(UV_CACHE_DIR=str(build_root / 'uv-cache'), TMPDIR=str(build_root / 'tmp'))
    (build_root / 'tmp').mkdir()
    fork = 'https://github.com/LiAuto-DSR/transformers-avavla.git'
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'init'], env=build_env)
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'fetch', '--depth', '1', fork, AVA_TRANSFORMERS_REV], env=build_env)
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'checkout', '--detach', 'FETCH_HEAD'], env=build_env)
    revision = subprocess.check_output(['git', '-c', 'core.autocrlf=false', '-C', str(source), 'rev-parse', 'HEAD'], text=True).strip()
    if revision != AVA_TRANSFORMERS_REV:
        raise RuntimeError('AVA Transformers checkout revision mismatch')
    llama = source / 'src/transformers/models/llama/modeling_llama.py'
    if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_BASE_SHA:
        raise RuntimeError('Unpatched AVA Transformers source SHA mismatch')
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'apply', '--check', patch], env=build_env)
    _VLADE_AVA_BASE_RUN(['git', '-c', 'core.autocrlf=false', '-C', source, 'apply', patch], env=build_env)
    if hashlib.sha256(llama.read_bytes()).hexdigest() != AVA_INSTALLED_SHA:
        raise RuntimeError('Patched AVA Transformers source SHA mismatch')
    wheels = build_root / 'wheels'
    wheels.mkdir()
    _VLADE_AVA_BASE_RUN([UV, 'build', '--wheel', '--python', '3.11', '--out-dir', wheels, source], env=build_env)
    wheel_files = list(wheels.glob('transformers-*.whl'))
    if len(wheel_files) != 1:
        raise RuntimeError('Expected exactly one patched Transformers wheel')
    wheel = wheel_files[0]
    with zipfile.ZipFile(wheel) as archive:
        wheel_source = archive.read('transformers/models/llama/modeling_llama.py')
    if hashlib.sha256(wheel_source).hexdigest() != AVA_INSTALLED_SHA:
        raise RuntimeError('Built Transformers wheel does not contain the approved patch')

    upstream = CODE / 'third_party/ava_vla'
    dependencies = tomllib.loads((upstream / 'pyproject.toml').read_text())['project']['dependencies']
    dependencies = [dep for dep in dependencies if not dep.startswith(('transformers @', 'dlimp @', 'huggingface_hub', 'accelerate'))]
    dependencies += [
        str(wheel),
        'dlimp @ git+https://github.com/moojink/dlimp_openvla.git@040105d256bd28866cc6620621a3d5f7b6b91b46',
        'protobuf==4.25.3', 'numpy==1.26.4', 'tensorflow-metadata==1.15.0', 'bitsandbytes==0.43.3',
        'huggingface_hub==0.36.2', 'accelerate==1.15.0']
    candidate_python = environment(('vlade_ava_fft_py311_' + RUN_ID), '3.11', dependencies)
    if str(candidate_python) != str(AVA_VENV / 'bin/python'):
        raise RuntimeError('Refusing to install outside the dedicated AVA environment')
    _VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python,
                        '--no-deps', '--force-reinstall', wheel], env=build_env)
    _VLADE_AVA_BASE_RUN([UV, 'pip', 'install', '--python', candidate_python, '--no-deps', '-e', upstream])
    verify_ava_transformers(candidate_python)
    _VLADE_AVA_BASE_RUN([UV,'pip','check','--python',candidate_python])
    cpu_import_env = ENV.copy()
    cpu_import_env['CUDA_VISIBLE_DEVICES'] = ''
    _VLADE_AVA_BASE_RUN([candidate_python,'-c',
        "import sys,importlib,json;assert sys.version_info[:2]==(3,11);names=['torch','numpy','transformers','accelerate','peft','bitsandbytes','draccus','dlimp','tensorflow','tensorflow_datasets','tensorflow_graphics','tensorflow_metadata','google.protobuf'];[importlib.import_module(n) for n in names];print(json.dumps(dict(cpu_imports='passed',modules=names,python=sys.version)))"],env=cpu_import_env)
    _VLADE_AVA_BASE_RUN([candidate_python, '-c', "import torch,transformers,tensorflow_datasets; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,transformers.__version__,torch.cuda.get_device_name(0),torch.cuda.get_device_properties(0).total_memory/2**30)"])
    freeze = subprocess.check_output([str(UV), 'pip', 'freeze', '--python', str(candidate_python)], text=True)
    (RUN_DIR / 'manifests/ava_environment_freeze.txt').write_text(freeze)
    (AVA_VENV / 'vlade_transformers_patch.json').write_text(json.dumps({
        'base_commit': AVA_TRANSFORMERS_REV, 'patch_sha256': AVA_PATCH_SHA,
        'installed_source_sha256': AVA_INSTALLED_SHA,
        'wheel_sha256': hashlib.sha256(wheel.read_bytes()).hexdigest()}, indent=2))
    AVA_PY = candidate_python
    AVA_ENV_READY = True
    print('AVA managed Transformers patch verified; FFT environment ready')
    control_write(RUN_DIR/'manifests/environment.json',{
      'python':str(AVA_PY),'patch_sha256':AVA_PATCH_SHA,'transformers_revision':AVA_TRANSFORMERS_REV,
      'installed_transformers_sha256':AVA_INSTALLED_SHA,'freeze_sha256':digest(RUN_DIR/'manifests/ava_environment_freeze.txt')})
else:
    print('Python 환경 준비 대기; Colab 기본 Python으로 AVA를 실행하지 않습니다.')

## 로컬 데이터 전체 무결성 및 실제 Processor/Collator
아래 셀은 학습을 실행하지 않습니다. 최초에는 TAR SHA·로컬 압축 해제·전체 payload SHA 검사를 수행합니다. 이후에는 파일 동일성 근거를 확인한 검증 기록을 재사용하고 사용한 검증 모드를 기록합니다.
고정 prepared manifest SHA는 이번 작업에서 기존 로컬 canonical D 보관본을 읽어 기록한 값입니다.
이 값의 신뢰 근거가 불충분하거나 파일이 다르면 변경해서 우회하지 말고 검증된 준비 기록을 확인하세요.
T0 weight/component는 기존 download manifest의 해시를 확인합니다.
공유 checkpoint의 변경 이력이 있는 config/Python 파일은 로컬에 읽기 전용으로 복사하고 실제 SHA를 기록합니다.
모델 자체의 로딩·forward/backward·재로딩 성공은 파일럿에서 별도로 확인합니다.

In [ ]:
if PREPARE_INPUTS:
    require_ava_environment()
    run([AVA_PY,'-c',"from pathlib import Path;import sys;from vlade.training.colab_suite_fft import runtime_api_report,write;write(Path(sys.argv[2]),runtime_api_report(Path(sys.argv[1])))",CODE,RUN_DIR/'manifests/runtime_api.json'])
    print((RUN_DIR/'manifests/runtime_api.json').read_text())
    from vlade.storage.assets import asset_record
    PREPARED=hf_input('prepared_d_seed42',AVA_PY)
    T0=hf_input('ava_t0',AVA_PY)
    map_path=PREPARED/'suite_map.json'
    if not map_path.is_file():
        map_root=hf_input(SUITE_MAP_ASSET,AVA_PY)
        map_path=map_root/'suite_map.json'
    if not map_path.is_file(): raise FileNotFoundError('Publish the original audited suite_map.json to HF first; labels are never inferred.')
    SUITE_MAP=LOCAL/'suite_map.json'
    if SUITE_MAP.exists() and digest(SUITE_MAP)!=digest(map_path):raise ValueError('Resume suite map changed')
    if not SUITE_MAP.exists():shutil.copyfile(map_path,SUITE_MAP)
    prep_script=LOCAL/'prepare_inputs.py'
    prep_script.write_text("from pathlib import Path\nimport sys\nfrom vlade.training.colab_suite_fft import verify_data, write\nwrite(Path(sys.argv[4]), verify_data(Path(sys.argv[2]), Path(sys.argv[3])))\n")
    data_report=RUN_DIR/'manifests/dataset_full_validation.json'
    run([AVA_PY,prep_script,'verify',PREPARED,SUITE_MAP,data_report])
    DATA_INFO=json.loads(data_report.read_text())
    DATA_INFO['verification_mode']='pinned_hf_archive_and_full_payload_validation'
    control_write(LOCAL/'prepared.json',{'prepared':str(PREPARED)})
    control_write(RUN_DIR/'manifests/dataset_manifest.json',DATA_INFO)
    control_write(RUN_DIR/'manifests/hf_inputs.json',{
        'prepared':asset_record('prepared_d_seed42'),'teacher':asset_record('ava_t0')})
    config = json.loads((CODE/'configs/experiments/libero_plus_fft.json').read_text())
    total_steps = math.ceil(228908*MAX_EPOCHS/(BATCH_SIZE*GRAD_ACCUMULATION_STEPS))
    config['fft_defaults'].update(prepared_data_root=str(PREPARED),data_root=str(PREPARED),
       image_aug=False,batch_size=BATCH_SIZE,grad_accumulation_steps=GRAD_ACCUMULATION_STEPS,
       seed=42,max_steps=total_steps,save_steps=SAVE_STEPS,save_optimizer_state=SAVE_OPTIMIZER_STATE,
       eval_steps=SAVE_STEPS,max_eval_batches=16,log_steps=1,debug_records=0,
       lr_schedule='constant',warmup_steps=0)
    config['fft_runs']['ava_vla'].update(checkpoint=str(T0),output_dir=str(RUN_DIR/'checkpoints'))
    MAIN_CONFIG = RUN_DIR/'configs/effective_config.json'
    if MAIN_CONFIG.exists() and json.loads(MAIN_CONFIG.read_text()) != config:
        raise ValueError('기존 본 학습 설정과 다릅니다. 덮어쓰지 않습니다.')
    control_write(MAIN_CONFIG,config)
    pilot_config = json.loads(json.dumps(config))
    pilot_config['fft_defaults'].update(max_steps=4,save_steps=4,eval_steps=4)
    pilot_config['fft_runs']['ava_vla']['output_dir']=str(RUN_DIR/'pilot4/checkpoints')
    PILOT_CONFIG = RUN_DIR/'configs/pilot4.json'
    if PILOT_CONFIG.exists() and json.loads(PILOT_CONFIG.read_text()) != pilot_config: raise ValueError('기존 pilot 설정 충돌')
    control_write(PILOT_CONFIG,pilot_config)
    inspect_script = LOCAL/'inspect_batch.py'
    inspect_script.write_text("""
import sys, torch
from pathlib import Path
from vlade.training import fft
from vlade.training.colab_suite_fft import install_hooks
from vlade.training.run_control import configure_tensorflow_cpu
configure_tensorflow_cpu(42)
fft._activate_upstream('ava_vla')
install_hooks(Path(sys.argv[2]))
cfg=fft.FFTConfig.from_json(Path(sys.argv[1]),'ava_vla')
fft.preflight(cfg)
from transformers import AutoConfig, AutoImageProcessor, AutoProcessor
from prismatic.extern.hf.configuration_prismatic import OpenVLAConfig
from prismatic.extern.hf.processing_prismatic import PrismaticImageProcessor, PrismaticProcessor
AutoConfig.register('openvla',OpenVLAConfig)
AutoImageProcessor.register(OpenVLAConfig,PrismaticImageProcessor)
AutoProcessor.register(OpenVLAConfig,PrismaticProcessor)
p=AutoProcessor.from_pretrained(cfg.checkpoint,trust_remote_code=True,local_files_only=True)
for train in (True,False):
    batch=next(iter(fft._make_loader(cfg,None,p,train=train)[0]))
    assert batch['actions'].shape[-2:]==(8,7)
    assert batch['proprio'].shape[-1]==8 and torch.isfinite(batch['actions']).all()
    assert batch['action_valid_mask'].shape[-2:]==(8,7)
    print('Actual processor/collator:', 'train suite_balanced' if train else 'validation episode',
          {k:list(v.shape) for k,v in batch.items() if hasattr(v,'shape')})
""")
    run([AVA_PY,inspect_script,MAIN_CONFIG,SUITE_MAP])
    READY=True
    print('HF inputs, full dataset validation and actual CPU collator verified. Training has not started.')


## 4-step 파일럿 · 저장과 재로딩
Colab UI에서 직전 CU를 읽어 `CU_BEFORE_PILOT`에 입력하고 RUN_PILOT=True로 실행합니다.
상한과 Drive 실제 여유 공간도 필요합니다. 모델·8-bit Optimizer 상태 저장 예상량은 T0 크기의 4배로
보수적으로 계산하고 파일럿 후 실제 checkpoint 크기로 대체합니다. 파일럿 가중치는 본 학습에 자동 사용되지 않습니다.

In [ ]:
def drive_free():
    return shutil.disk_usage(OUTPUT_ROOT).free

def launch(settings, name):
    path=RUN_DIR/'configs'/name
    control_write(path,settings)
    run([AVA_PY,'-m','vlade.training.colab_suite_fft',path])
def reload_check(checkpoint, config_path, report):
    script=LOCAL/'reload_check.py'
    script.write_text("""
import sys, runpy
from pathlib import Path
from vlade.training.colab_suite_fft import install_hooks
install_hooks(Path(sys.argv[1]))
sys.argv=['check_fft_checkpoint.py',*sys.argv[2:]]
runpy.run_path('scripts/train/check_fft_checkpoint.py',run_name='__main__')
""")
    run([AVA_PY,script,SUITE_MAP,'--checkpoint',checkpoint,'--data',PREPARED,
         '--config',config_path,'--report',report])
if RUN_PILOT:
    if not READY: raise RuntimeError('먼저 준비 및 실제 batch 검사를 완료하세요.')
    require_ava_environment()
    if CU_BEFORE_PILOT is None or CU_BEFORE_PILOT<=TARGET_REMAINING_CU:
        raise ValueError('파일럿 직전의 실제 CU를 입력하세요.')
    if PILOT_TIME_LIMIT_SECONDS is None or PILOT_TIME_LIMIT_SECONDS<=600:
        raise ValueError('파일럿 로딩·4-step·저장·재로딩을 포함하는 명시적 시간 상한(600초 초과)이 필요합니다.')
    checkpoint_estimate = 4*sum(p.stat().st_size for p in T0.iterdir() if p.suffix in {'.pt','.safetensors'})
    if drive_free()<checkpoint_estimate*1.35: raise ValueError('파일럿 checkpoint 저장 공간 부족')
    PILOT_STARTED_UNIX=time.time()
    pilot_deadline=PILOT_STARTED_UNIX+PILOT_TIME_LIMIT_SECONDS
    if SESSION_DEADLINE_UNIX is not None: pilot_deadline=min(pilot_deadline,SESSION_DEADLINE_UNIX)
    launch(dict(authorized=True,phase='pilot4',config=str(PILOT_CONFIG),run_root=str(RUN_DIR),
       suite_map=str(SUITE_MAP),deadline_unix=pilot_deadline,reserve_seconds=300,stop_step=4,
       free_bytes_at_start=DRIVE_FREE_BYTES_AT_START,checkpoint_bytes=checkpoint_estimate),'pilot4_launch.json')
    pilot=json.loads((RUN_DIR/'pilot4/pilot_report.json').read_text())
    if pilot['step']!=4 or not pilot['training_complete'] or pilot['validation_seconds']<=0:
        raise RuntimeError('파일럿 4-step 또는 Validation 미완료; 본 학습 금지')
    before=time.time()
    reload_check(Path(pilot['checkpoint']),PILOT_CONFIG,RUN_DIR/'pilot4/reload_report.json')
    pilot['reload_seconds']=time.time()-before
    pilot['measurement_started_unix']=PILOT_STARTED_UNIX
    pilot['measurement_ended_unix']=time.time()
    pilot['cu_before']=CU_BEFORE_PILOT
    control_write(RUN_DIR/'pilot4/pilot_report.json',pilot)
    print('파일럿 완료. Colab UI의 종료 후 CU를 CU_AFTER_PILOT에 입력하세요.')
else:
    print('파일럿 미실행: RUN_PILOT=False')

## 파일럿 기반 예산 · 본 학습 명시적 실행
이 셀 직전에 `CU_AFTER_PILOT`과 `CURRENT_COMPUTE_UNITS`를 갱신하세요.
CU 측정 구간은 파일럿 직전부터 저장/재로딩 종료 직후까지입니다. 세션/수동 상한은 계획 시점의 잔여량입니다.
전체 목표 Step은 고정하여 cosine/attention 스케줄과 Resume 의미를 바꾸지 않습니다.
세션에서 가능한 Step은 별도로 중단합니다. 2000 미만이어도 마지막 checkpoint를 저장합니다.
재개는 명시한 동일 Run의 `checkpoints/step-XXXXXXX`만 허용합니다.

In [ ]:
BUDGET=None
if READY and (RUN_DIR/'pilot4/pilot_report.json').is_file():
    require_ava_environment()
    pilot=json.loads((RUN_DIR/'pilot4/pilot_report.json').read_text())
    reload_report=json.loads((RUN_DIR/'pilot4/reload_report.json').read_text())
    if reload_report['status']!='passed' or pilot['step']!=4: raise ValueError('파일럿 재로딩 증거 없음')
    probe=LOCAL/'pilot_context_check.json'
    run([AVA_PY,'-c',"import sys,torch;from pathlib import Path;from vlade.training.colab_suite_fft import install_hooks,write;write(Path(sys.argv[2]),dict(fingerprint=install_hooks(Path(sys.argv[1]))['fingerprint'],gpu=dict(gpu=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory)))",SUITE_MAP,probe])
    observed=json.loads(probe.read_text())
    if observed['fingerprint']!=pilot['fingerprint'] or observed['gpu']!=pilot['gpu']:
        raise ValueError('파일럿과 현재 소스/환경/Teacher/GPU/VRAM이 다릅니다. 새 파일럿 필요')
    profile={k:pilot[k] for k in ['step_seconds','startup_seconds','reload_seconds','validation_seconds']}
    # 첫 step의 모델 warm-up/전체 Query 인덱스 준비 비용은 별도 startup으로 반영합니다.
    profile['startup_seconds']+=max(0,pilot['first_step_seconds']-profile['step_seconds'])
    profile['checkpoint_seconds']=max(x['save_verify_seconds'] for x in pilot['saves'])
    checkpoint_bytes=max(x['bytes'] for x in pilot['saves'])
    resume_step=0
    if RESUME_CHECKPOINT:
        resume=Path(RESUME_CHECKPOINT)
        if not resume.resolve().is_relative_to((RUN_DIR/'checkpoints').resolve()): raise ValueError('다른 Run/pilot 재개 금지')
        resume_step=json.loads((resume/'vlade_fft.json').read_text())['step']
    plan_args=dict(profile=profile,current_cu=CURRENT_COMPUTE_UNITS,target_cu=TARGET_REMAINING_CU,
       cu_before=pilot['cu_before'],cu_after=CU_AFTER_PILOT,
       cu_elapsed=pilot['measurement_ended_unix']-pilot['measurement_started_unix'],
       session_seconds=None if SESSION_DEADLINE_UNIX is None else SESSION_DEADLINE_UNIX-time.time(),
       explicit_seconds=EXPLICIT_MAIN_TIME_LIMIT_SECONDS,free_bytes=drive_free(),
       checkpoint_bytes=checkpoint_bytes,remaining_steps=total_steps-resume_step,save_steps=SAVE_STEPS,
       resume_step=resume_step)
    # 명시적 시간 모드 선택: CU 실측이 둘 다 없을 때만 소모율을 생략합니다.
    if CU_AFTER_PILOT is None: plan_args.update(cu_before=None,cu_after=None)
    plan_input=LOCAL/'plan_input.json'; plan_output=LOCAL/'plan_output.json'
    control_write(plan_input,plan_args)
    run([AVA_PY,'-c',"from pathlib import Path;import sys;from vlade.training.colab_suite_fft import read,write,plan_budget;write(Path(sys.argv[2]),plan_budget(**read(Path(sys.argv[1]))))",plan_input,plan_output])
    BUDGET=json.loads(plan_output.read_text())
    control_write(RUN_DIR/'reports/budget.json',BUDGET)
    print(json.dumps(BUDGET,indent=2,ensure_ascii=False))
    if not BUDGET['cu_target_supported']: print('명시적 시간 모드: 5 CU 잔여 목표 제어 불가')
if RUN_MAIN_FFT:
    try:
        if BUDGET is None: raise RuntimeError('현재 조건의 파일럿 기반 예산이 필요합니다.')
        main_started=time.time()
        main_deadline=main_started+BUDGET['wall_seconds']
        launch(dict(authorized=True,phase='main',config=str(MAIN_CONFIG),run_root=str(RUN_DIR),
          suite_map=str(SUITE_MAP),resume=RESUME_CHECKPOINT,
          deadline_unix=main_deadline,reserve_seconds=BUDGET['save_reserve_seconds'],
          stop_step=resume_step+BUDGET['steps_this_session'],step_seconds=profile['step_seconds'],budget=BUDGET,
          first_step_seconds=pilot['first_step_seconds'],
          free_bytes_at_start=DRIVE_FREE_BYTES_AT_START,checkpoint_bytes=checkpoint_bytes),'main_launch.json')
        summary=json.loads((RUN_DIR/'reports/run_summary.json').read_text())
        if summary['status'] not in {'completed','budget_exhausted'} or not summary['checkpoint_verified']:
            raise RuntimeError('정상 저장 확인 실패; 런타임을 해제하지 않습니다.')
        # Re-read the checkpoint seal and all payloads in a separate process before unassign.
        run([AVA_PY,'-c',"from pathlib import Path;import sys;from vlade.training.run_control import verify_checkpoint;print(verify_checkpoint(Path(sys.argv[1]))['step'])",summary['checkpoint']])
        summary['checkpoint_readback_verified']=True
        summary['elapsed_seconds_including_final_readback']=time.time()-main_started
        rate=BUDGET['measured_cu_per_second']
        if rate is not None:
            summary['estimated_cu_used']=rate*summary['elapsed_seconds_including_final_readback']
            summary['estimated_remaining_cu']=CURRENT_COMPUTE_UNITS-summary['estimated_cu_used']
        summary['cu_estimate_excludes_runtime_release']=True
        control_write(RUN_DIR/'reports/run_summary.json',summary)
        print(json.dumps(summary,indent=2,ensure_ascii=False))
        if AUTO_UNASSIGN_RUNTIME:
            from google.colab import runtime
            # 오류 시 unassign에 도달하지 않습니다. 공유 원본에는 쓰지 않고 이 Run의 저장을 flush합니다.
            remaining=main_deadline-time.time()
            if remaining<=0: raise RuntimeError('Checkpoint readback 예비시간 초과; 자동 해제를 하지 않습니다.')

            runtime.unassign()
    except BaseException:
        import traceback
        traceback.print_exc()
        error=traceback.format_exc()
        (LOCAL/'main_failure.log').write_text(error)
        if RUN_DIR.exists():
            with (RUN_DIR/'logs/errors.log').open('a') as stream: stream.write(error)
        raise

else:
    print('본 학습 미실행: RUN_MAIN_FFT=False. 가능한 Step/CU/용량은 파일럿 및 사용자 입력 전에는 미확정입니다.')

## 재시작 및 한계
기존 Run ID와 명시적 본 학습 checkpoint를 지정하고 로컬 준비를 다시 수행합니다.
기존 고정 설정이 다르면 중단합니다. batch/accumulation/seed/샘플링·suite map·소스·정규화·Optimizer
호환성을 검사합니다. Query cursor는 `step × accumulation × batch`를 같은 seed로 재생/skip합니다.
num_workers=0, image_aug=false를 전제로 하며 GPU의 bitwise 일치까지 보장하지 않습니다.
파일럿과 GPU/VRAM이 바뀌면 해당 환경의 새 파일럿이 필요합니다.
오류 및 저장 실패는 traceback/stderr를 남기며 런타임 자동 해제를 수행하지 않습니다.
Drive readback 후 `drive.flush_and_unmount()`가 성공한 경우에만 unassign합니다.
이는 Drive 서버의 영구 내구성을 보장한다는 의미는 아닙니다.

입력은 Hugging Face 고정 revision에서 받습니다. 다운로드·캐시 재사용 시 파일 무결성을 검사합니다. 소스는 실행 시 확정한 Git commit을 기록합니다.